In [ ]:
# %% [markdown]
# ##  Step 1: Setup

# %%
!pip install -q imbalanced-learn optuna xgboost lightgbm catboost

import gc
import os
import time
import random
import shutil
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

def clear_mem():
    gc.collect()

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# Traditional ML
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Ensemble ML
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    BaggingClassifier, HistGradientBoostingClassifier,
    VotingClassifier
)

# Advanced Boosting
try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
    print("✅ XGBoost")
except:
    XGBOOST_OK = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_OK = True
    print("✅ LightGBM")
except:
    LIGHTGBM_OK = False

try:
    from catboost import CatBoostClassifier
    CATBOOST_OK = True
    print("✅ CatBoost")
except:
    CATBOOST_OK = False

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU available")

try:
    import optuna
    OPTUNA_OK = True
except:
    OPTUNA_OK = False

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("\n✅ Setup complete!")

# %% [markdown]
# ##  Step 2: Mount Drive with Robust Copy Function

# %%
from google.colab import drive

# Paths
DRIVE_DATA_DIR = "/content/drive/MyDrive/Dataset/csv/CICIoT2023"
LOCAL_DATA_DIR = "/content/ciciot_local"
OUT_DIR = "/content/outputs_final"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

def mount_drive():
    """Mount or remount Google Drive"""
    try:
        drive.mount('/content/drive', force_remount=True)
        return True
    except Exception as e:
        print(f"⚠️ Mount failed: {e}")
        return False

def copy_single_file(src, dst, max_retries=3):
    """Copy a single file with retry logic"""
    for attempt in range(max_retries):
        try:
            if os.path.exists(dst):
                return True  # Already exists

            # Try copying
            shutil.copy2(src, dst)
            return True

        except OSError as e:
            print(f"      ⚠️ Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                print("      🔄 Remounting Drive...")
                time.sleep(2)
                mount_drive()
                time.sleep(2)
            else:
                return False
    return False

def get_local_files():
    """Get list of CSV files already in local storage"""
    if os.path.exists(LOCAL_DATA_DIR):
        return [os.path.join(LOCAL_DATA_DIR, f)
                for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.csv')]
    return []

# Mount drive
print("📂 Mounting Google Drive...")
mount_drive()

# Check if we already have local files from previous run
existing_local = get_local_files()
if existing_local:
    print(f"✅ Found {len(existing_local)} files already in local storage!")
    for f in existing_local:
        size = os.path.getsize(f) / 1e6
        print(f"   - {os.path.basename(f)} ({size:.1f} MB)")

# %% [markdown]
# ##  Step 3: Configuration

# %%
CONFIG = {
    'n_files_to_copy': 5,
    'target_rows': 100_000,
    'min_class_count': 200,
    'test_size': 0.20,
    'batch_size': 256,
    'epochs_dl': 25,
    'epochs_kd': 15,
    'patience': 5,
    'kd_temp': 4.0,
    'kd_alpha': 0.3,
    'optuna_trials': 10,
}
print("📋 Config:", CONFIG)

# %% [markdown]
# ##  Step 4: Copy Files (with Retry) or Use Existing

# %%
def ensure_local_files(drive_dir, local_dir, n_files_needed):
    """Ensure we have enough local files, copying if necessary"""

    # Check existing
    existing = get_local_files()
    if len(existing) >= n_files_needed:
        print(f"✅ Using {len(existing)} existing local files")
        return existing[:n_files_needed]

    # Need to copy more
    print(f"📥 Need to copy files (have {len(existing)}, need {n_files_needed})")

    if not os.path.exists(drive_dir):
        print(f"❌ Drive path not found: {drive_dir}")
        print("   Please update DRIVE_DATA_DIR!")
        return existing  # Return whatever we have

    # Get list of files to copy
    all_drive_files = sorted([f for f in os.listdir(drive_dir) if f.endswith('.csv')])
    existing_names = set(os.path.basename(f) for f in existing)
    files_to_copy = [f for f in all_drive_files if f not in existing_names]

    # Select files spread across dataset
    n_to_copy = n_files_needed - len(existing)
    step = max(1, len(files_to_copy) // n_to_copy)
    selected = [files_to_copy[i] for i in range(0, len(files_to_copy), step)][:n_to_copy]

    print(f"   Attempting to copy {len(selected)} files...")

    copied = 0
    for i, fname in enumerate(selected):
        src = os.path.join(drive_dir, fname)
        dst = os.path.join(local_dir, fname)

        print(f"   [{i+1}/{len(selected)}] {fname}...")

        if copy_single_file(src, dst):
            size = os.path.getsize(dst) / 1e6
            print(f"       ✅ Done ({size:.1f} MB)")
            copied += 1
        else:
            print(f"       ❌ Failed - skipping")

        # Small delay between files
        time.sleep(1)

    print(f"\n✅ Successfully copied {copied} new files")
    return get_local_files()

# Get files
local_files = ensure_local_files(DRIVE_DATA_DIR, LOCAL_DATA_DIR, CONFIG['n_files_to_copy'])

if len(local_files) == 0:
    print("\n❌ NO FILES AVAILABLE!")
    print("Please manually upload some CSV files to /content/ciciot_local/")
    print("Or fix the Drive path and re-run")
else:
    print(f"\n✅ Ready with {len(local_files)} files")

# %% [markdown]
# ##  Step 5: Load Data

# %%
def normalize_labels(s):
    return s.astype(str).str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

def add_features(df, cols):
    new = []
    if 'Tot sum' in df.columns and 'Number' in df.columns:
        df['bytes_pkt'] = df['Tot sum'] / (df['Number'] + 1)
        new.append('bytes_pkt')
    if 'Srate' in df.columns and 'Drate' in df.columns:
        df['rate_ratio'] = df['Srate'] / (df['Drate'] + 0.001)
        new.append('rate_ratio')
    for c in ['flow_duration', 'Tot sum']:
        if c in df.columns:
            df[f'{c}_log'] = np.log1p(df[c].clip(lower=0))
            new.append(f'{c}_log')
    for c in new:
        df[c] = df[c].replace([np.inf, -np.inf], 0).fillna(0)
    return df, cols + new

def load_data(files, config):
    """Load data from local files"""
    print(f"\n📂 Loading from {len(files)} local files...")

    if len(files) == 0:
        raise ValueError("No files to load!")

    df0 = pd.read_csv(files[0], nrows=5)
    label_col = [c for c in df0.columns if c.lower() == 'label'][0]
    feat_cols = [c for c in df0.columns if c != label_col]

    # Scan for labels
    all_labels = set()
    for f in files[:min(3, len(files))]:
        try:
            s = pd.read_csv(f, usecols=[label_col], nrows=5000)[label_col]
            all_labels |= set(normalize_labels(s).unique())
        except:
            pass

    cap = max(800, config['target_rows'] // max(5, len(all_labels)))
    print(f"   Cap per class: {cap}")

    X_list, y_list, counts = [], [], {}

    for fpath in files:
        fname = os.path.basename(fpath)
        try:
            df = pd.read_csv(fpath, usecols=feat_cols + [label_col])
            print(f"   ✅ {fname}: {len(df):,} rows")

            y = normalize_labels(df[label_col])
            df, feat_new = add_features(df, feat_cols)
            X = df[feat_new].apply(pd.to_numeric, errors='coerce')
            mask = X.notna().all(axis=1) & y.notna()
            X, y = X.loc[mask], y.loc[mask]

            for lab in y.unique():
                if counts.get(lab, 0) >= cap:
                    continue
                idx = y.index[y == lab]
                take = min(cap - counts.get(lab, 0), len(idx))
                if take > 0:
                    chosen = np.random.choice(idx, take, replace=False)
                    X_list.append(X.loc[chosen].values.astype(np.float32))
                    y_list.append(y.loc[chosen].values)
                    counts[lab] = counts.get(lab, 0) + take

            del df, X, y
            clear_mem()

            if sum(counts.values()) >= config['target_rows']:
                break

        except Exception as e:
            print(f"   ⚠️ {fname}: {e}")

    if len(X_list) == 0:
        raise ValueError("No data loaded!")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    del X_list, y_list
    clear_mem()

    print(f"\n✅ Loaded: {X.shape[0]:,} samples, {X.shape[1]} features, {len(np.unique(y))} classes")
    return X, y, feat_new

# Load
X_all, y_all, feature_names = load_data(local_files, CONFIG)

# %% [markdown]
# ##  Step 6: Preprocess

# %%
# Filter rare classes
vc = pd.Series(y_all).value_counts()
print(f"📊 Top classes:")
print(vc.head(10))

keep = set(vc[vc >= CONFIG['min_class_count']].index)
mask = np.array([y in keep for y in y_all])
X_all, y_all = X_all[mask], y_all[mask]

# Encode
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
classes = list(le.classes_)
n_classes = len(classes)
print(f"\n✅ {len(X_all):,} samples, {n_classes} classes")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_enc, test_size=CONFIG['test_size'], random_state=SEED, stratify=y_enc)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.12, random_state=SEED, stratify=y_train)

del X_all, y_all, y_enc
clear_mem()

print(f"   Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

del X_train, X_val, X_test
clear_mem()

# Class weights
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

# Balance
def balance(X, y, cap=5000):
    Xb, yb = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        n = min(cap, len(idx))
        ch = np.random.choice(idx, n, replace=(len(idx) < n))
        Xb.append(X[ch])
        yb.append(y[ch])
    Xo, yo = np.vstack(Xb).astype(np.float32), np.concatenate(yb)
    perm = np.random.permutation(len(yo))
    return Xo[perm], yo[perm]

X_bal, y_bal = balance(X_train_s, y_train, 5000)
print(f"✅ Balanced: {len(X_bal):,}")

joblib.dump(le, f"{OUT_DIR}/label_encoder.joblib")
joblib.dump(scaler, f"{OUT_DIR}/scaler.joblib")

# %% [markdown]
# ##  Helper Functions

# %%
def metrics(y_true, y_pred, y_proba=None, name="Model", t=0):
    m = OrderedDict([
        ('Model', name),
        ('Accuracy', accuracy_score(y_true, y_pred)),
        ('Balanced_Acc', balanced_accuracy_score(y_true, y_pred)),
        ('Macro_Precision', precision_score(y_true, y_pred, average='macro', zero_division=0)),
        ('Macro_Recall', recall_score(y_true, y_pred, average='macro', zero_division=0)),
        ('Macro_F1', f1_score(y_true, y_pred, average='macro', zero_division=0)),
        ('Weighted_F1', f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        ('MCC', matthews_corrcoef(y_true, y_pred)),
        ('Cohen_Kappa', cohen_kappa_score(y_true, y_pred)),
        ('Time_s', round(t, 1)),
    ])
    if y_proba is not None:
        try:
            m['ROC_AUC'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
        except:
            m['ROC_AUC'] = np.nan
    return m

def save_fig(path, dpi=150):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"📊 Saved: {os.path.basename(path)}")

# %% [markdown]
# ---
# #  BENCHMARK (28+ Models)
# ---

# %% [markdown]
# ## 🏛️ Category 1: Traditional ML (8 models)

# %%
print("\n" + "="*70)
print("🏛️ CATEGORY 1: TRADITIONAL ML (8 models)")
print("="*70)

results = []

# Subsample for slow models
n_sub = min(25000, len(X_bal))
idx_sub = np.random.choice(len(X_bal), n_sub, replace=False)
X_sub, y_sub = X_bal[idx_sub], y_bal[idx_sub]

trad_models = [
    ("Logistic Regression", LogisticRegression(max_iter=300, n_jobs=-1, random_state=SEED), False),
    ("Ridge Classifier", RidgeClassifier(random_state=SEED), False),
    ("SGD Classifier", SGDClassifier(loss='log_loss', max_iter=100, n_jobs=-1, random_state=SEED), False),
    ("Perceptron", Perceptron(max_iter=100, n_jobs=-1, random_state=SEED), False),
    ("Gaussian Naive Bayes", GaussianNB(), False),
    ("LDA", LinearDiscriminantAnalysis(), False),
    ("KNN (k=5)", KNeighborsClassifier(n_neighbors=5, n_jobs=-1), True),  # Use subsample
    ("Decision Tree", DecisionTreeClassifier(max_depth=20, random_state=SEED), False),
]

for name, clf, use_sub in trad_models:
    print(f"\n🔄 {name}")
    t0 = time.time()

    if use_sub:
        clf.fit(X_sub, y_sub)
    else:
        clf.fit(X_bal, y_bal)

    t = time.time() - t0
    pred = clf.predict(X_test_s)
    proba = clf.predict_proba(X_test_s) if hasattr(clf, 'predict_proba') else None

    m = metrics(y_test, pred, proba, name, t)
    m['Category'] = 'Traditional ML'
    results.append(m)
    print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

    del clf
    clear_mem()

# %% [markdown]
# ##  Category 2: Ensemble ML (9 models)

# %%
print("\n" + "="*70)
print("🌲 CATEGORY 2: ENSEMBLE ML (9 models)")
print("="*70)

trained = {}

ensemble_models = [
    ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=SEED)),
    ("Extra Trees", ExtraTreesClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=SEED)),
    ("Bagging (DT)", BaggingClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)),
    ("AdaBoost", AdaBoostClassifier(n_estimators=100, random_state=SEED)),
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, max_depth=8, random_state=SEED)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=200, max_depth=12, random_state=SEED)),
]

if XGBOOST_OK:
    ensemble_models.append(("XGBoost", XGBClassifier(
        n_estimators=200, max_depth=10, learning_rate=0.1,
        n_jobs=-1, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='mlogloss'
    )))

if LIGHTGBM_OK:
    ensemble_models.append(("LightGBM", LGBMClassifier(
        n_estimators=200, max_depth=12, learning_rate=0.1,
        n_jobs=-1, random_state=SEED, verbose=-1
    )))

if CATBOOST_OK:
    ensemble_models.append(("CatBoost", CatBoostClassifier(
        iterations=200, depth=8, learning_rate=0.1,
        random_state=SEED, verbose=0
    )))

for name, clf in ensemble_models:
    print(f"\n🔄 {name}")
    t0 = time.time()
    clf.fit(X_bal, y_bal)
    t = time.time() - t0

    pred = clf.predict(X_test_s)
    proba = clf.predict_proba(X_test_s)

    m = metrics(y_test, pred, proba, name, t)
    m['Category'] = 'Ensemble ML'
    results.append(m)
    trained[name] = clf
    print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

clear_mem()

# %% [markdown]
# ##  Category 3: Deep Learning (6 models)

# %%
print("\n" + "="*70)
print("🧠 CATEGORY 3: DEEP LEARNING (6 models)")
print("="*70)

input_dim = X_bal.shape[1]
cbs = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['patience'], restore_best_weights=True)]

dl_results = []

# 1. Shallow MLP
print("\n🔄 Shallow MLP")
m1 = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.3),
    layers.Dense(n_classes)
])
m1.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m1.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
       batch_size=CONFIG['batch_size'], callbacks=cbs, verbose=0)
logits = m1.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "Shallow MLP", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m1; clear_mem()

# 2. Deep MLP
print("\n🔄 Deep MLP")
m2 = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes)
])
m2.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m2.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
       batch_size=CONFIG['batch_size'], callbacks=cbs, class_weight=class_weights, verbose=0)
logits = m2.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "Deep MLP", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m2; clear_mem()

# 3. Wide MLP
print("\n🔄 Wide MLP")
m3 = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(n_classes)
])
m3.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m3.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
       batch_size=CONFIG['batch_size'], callbacks=cbs, verbose=0)
logits = m3.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "Wide MLP", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m3; clear_mem()

# 4. 1D-CNN
print("\n🔄 1D-CNN")
m4 = keras.Sequential([
    layers.Reshape((input_dim, 1), input_shape=(input_dim,)),
    layers.Conv1D(64, 3, padding='same', activation='relu'),
    layers.Conv1D(128, 3, padding='same', activation='relu'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes)
])
m4.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m4.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
       batch_size=CONFIG['batch_size'], callbacks=cbs, verbose=0)
logits = m4.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "1D-CNN", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m4; clear_mem()

# 5. LSTM
print("\n🔄 LSTM")
m5_in = keras.Input(shape=(input_dim,))
x = layers.Reshape((input_dim, 1))(m5_in)
x = layers.LSTM(64)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
m5_out = layers.Dense(n_classes)(x)
m5 = keras.Model(m5_in, m5_out)
m5.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m5.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=min(15, CONFIG['epochs_dl']),
       batch_size=CONFIG['batch_size'], callbacks=cbs, verbose=0)
logits = m5.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "LSTM", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m5; clear_mem()

# 6. TabNet-style (Attention)
print("\n🔄 Attention MLP")
m6_in = keras.Input(shape=(input_dim,))
x = layers.Dense(128)(m6_in)
x = layers.Reshape((128, 1))(x)
attn = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
x = layers.Add()([x, attn])
x = layers.LayerNormalization()(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
m6_out = layers.Dense(n_classes)(x)
m6 = keras.Model(m6_in, m6_out)
m6.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time()
m6.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
       batch_size=CONFIG['batch_size'], callbacks=cbs, verbose=0)
logits = m6.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "Attention MLP", time.time()-t0)
m['Category'] = 'Deep Learning'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f}")
del m6; clear_mem()

# %% [markdown]
# ##  Category 4: PROPOSED SOLUTION

# %%
print("\n" + "="*70)
print("🚀 CATEGORY 4: PROPOSED SOLUTION")
print("="*70)

# 4.1 Proposed Teacher
print("\n🔄 Proposed DL Teacher (GELU + Residual + BN)")
inp = keras.Input(shape=(input_dim,))
x = layers.Dense(512, kernel_initializer='he_normal')(inp)
x = layers.BatchNormalization()(x)
x = layers.Activation('gelu')(x)
x = layers.Dropout(0.3)(x)
x1 = layers.Dense(512, kernel_initializer='he_normal')(x)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Activation('gelu')(x1)
x1 = layers.Dropout(0.3)(x1)
x = layers.Add()([x, x1])
x = layers.Dense(256, activation='gelu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)
x = layers.Dense(128, activation='gelu')(x)
out = layers.Dense(n_classes)(x)
teacher = keras.Model(inp, out)

steps = CONFIG['epochs_dl'] * (len(X_bal) // CONFIG['batch_size'])
lr = keras.optimizers.schedules.CosineDecay(1e-3, steps, alpha=1e-5)
teacher.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

t0 = time.time()
teacher.fit(X_bal, y_bal, validation_data=(X_val_s, y_val), epochs=CONFIG['epochs_dl'],
            batch_size=CONFIG['batch_size'], callbacks=cbs, class_weight=class_weights, verbose=1)
logits = teacher.predict(X_test_s, verbose=0)
pred_teacher = np.argmax(logits, 1)
proba_teacher = tf.nn.softmax(logits).numpy()
m = metrics(y_test, pred_teacher, proba_teacher, "Proposed DL Teacher", time.time()-t0)
m['Category'] = 'Proposed'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

# 4.2 Tuned HistGB
if OPTUNA_OK:
    print(f"\n🔄 Optuna Tuning ({CONFIG['optuna_trials']} trials)")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def obj(trial):
        p = {
            'learning_rate': trial.suggest_float('lr', 0.02, 0.15),
            'max_depth': trial.suggest_int('d', 6, 18),
            'max_iter': trial.suggest_int('i', 100, 300),
        }
        clf = HistGradientBoostingClassifier(**p, random_state=SEED)
        clf.fit(X_bal, y_bal)
        return f1_score(y_val, clf.predict(X_val_s), average='macro')

    study = optuna.create_study(direction='maximize')
    study.optimize(obj, n_trials=CONFIG['optuna_trials'], show_progress_bar=True)
    print(f"   Best F1: {study.best_value:.4f}")

    tuned_hgb = HistGradientBoostingClassifier(
        learning_rate=study.best_params['lr'],
        max_depth=study.best_params['d'],
        max_iter=study.best_params['i'],
        random_state=SEED
    )
    t0 = time.time()
    tuned_hgb.fit(X_bal, y_bal)
    pred = tuned_hgb.predict(X_test_s)
    proba = tuned_hgb.predict_proba(X_test_s)
    m = metrics(y_test, pred, proba, "Proposed Tuned HistGB", time.time()-t0)
    m['Category'] = 'Proposed'
    results.append(m)
    trained['Tuned_HistGB'] = tuned_hgb
    print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

# 4.3 Voting Ensemble
print("\n🔄 Proposed Voting Ensemble")
ens_list = [
    ('hgb', trained.get('Tuned_HistGB', trained['HistGradientBoosting'])),
    ('rf', trained['Random Forest']),
    ('et', trained['Extra Trees']),
]
if 'XGBoost' in trained:
    ens_list.append(('xgb', trained['XGBoost']))
if 'LightGBM' in trained:
    ens_list.append(('lgbm', trained['LightGBM']))

ensemble = VotingClassifier(estimators=ens_list, voting='soft', n_jobs=-1)
t0 = time.time()
ensemble.fit(X_bal, y_bal)
pred_ens = ensemble.predict(X_test_s)
proba_ens = ensemble.predict_proba(X_test_s)
m = metrics(y_test, pred_ens, proba_ens, "Proposed Voting Ensemble", time.time()-t0)
m['Category'] = 'Proposed'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

# 4.4 KD Student
print("\n🔄 Proposed KD Student")
student = keras.Sequential([
    layers.Dense(256, activation='gelu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='gelu'),
    layers.Dropout(0.15),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes)
])

class Distiller(keras.Model):
    def __init__(self, s, t, temp, alpha):
        super().__init__()
        self.s, self.t, self.temp, self.alpha = s, t, temp, alpha
        self.loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    def compile(self, opt):
        super().compile()
        self.opt = opt
    def train_step(self, data):
        x, y = data
        t_log = tf.stop_gradient(self.t(x, training=False))
        with tf.GradientTape() as tape:
            s_log = self.s(x, training=True)
            hard = self.loss_fn(y, s_log)
            soft = keras.losses.KLDivergence()(tf.nn.softmax(t_log/self.temp), tf.nn.softmax(s_log/self.temp)) * self.temp**2
            loss = self.alpha * hard + (1-self.alpha) * soft
        grads = tape.gradient(loss, self.s.trainable_variables)
        self.opt.apply_gradients(zip(grads, self.s.trainable_variables))
        return {'loss': loss}

dist = Distiller(student, teacher, CONFIG['kd_temp'], CONFIG['kd_alpha'])
dist.compile(keras.optimizers.Adam(1e-3))
t0 = time.time()
dist.fit(X_bal, y_bal, epochs=CONFIG['epochs_kd'], batch_size=CONFIG['batch_size'], verbose=1)
logits = student.predict(X_test_s, verbose=0)
m = metrics(y_test, np.argmax(logits, 1), tf.nn.softmax(logits).numpy(), "Proposed KD Student", time.time()-t0)
m['Category'] = 'Proposed'
results.append(m)
print(f"   Acc: {m['Accuracy']:.4f} | F1: {m['Macro_F1']:.4f} | MCC: {m['MCC']:.4f}")

# %% [markdown]
# ---
# #  FINAL RESULTS
# ---

# %%
df = pd.DataFrame(results).sort_values('Macro_F1', ascending=False).reset_index(drop=True)
df.insert(0, 'Rank', range(1, len(df) + 1))

print("\n" + "="*100)
print(f"📊 FINAL BENCHMARK RESULTS ({len(df)} Models)")
print("="*100)
print(df[['Rank', 'Model', 'Category', 'Accuracy', 'Macro_F1', 'MCC', 'Time_s']].to_string(index=False))

df.to_csv(f"{OUT_DIR}/benchmark_results.csv", index=False)

# Summary
print("\n" + "="*70)
print("📊 CATEGORY SUMMARY")
print("="*70)
for cat in ['Traditional ML', 'Ensemble ML', 'Deep Learning', 'Proposed']:
    sub = df[df['Category'] == cat]
    if len(sub) > 0:
        print(f"\n{cat}:")
        print(f"   Best: {sub.iloc[0]['Model']} (F1={sub.iloc[0]['Macro_F1']:.4f})")
        print(f"   Avg F1: {sub['Macro_F1'].mean():.4f}")

# %% [markdown]
# ##  Visualizations

# %%
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
colors = {'Traditional ML': '#3498db', 'Ensemble ML': '#27ae60', 'Deep Learning': '#9b59b6', 'Proposed': '#e74c3c'}

for idx, (metric, title) in enumerate([('Accuracy', 'Accuracy'), ('Macro_F1', 'Macro F1'), ('MCC', 'MCC')]):
    ax = axes[idx]
    df_s = df.sort_values(metric, ascending=True)
    c = [colors[cat] for cat in df_s['Category']]
    ax.barh(range(len(df_s)), df_s[metric], color=c)
    ax.set_yticks(range(len(df_s)))
    ax.set_yticklabels(df_s['Model'], fontsize=8)
    ax.set_xlabel(title)
    ax.set_title(f'{title} Comparison', fontweight='bold')
    ax.set_xlim(0.4, 1.0)
    ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=c, label=l) for l, c in colors.items()],
           loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.02))
save_fig(f"{OUT_DIR}/comparison.png", dpi=200)
plt.show()

# Confusion Matrix
vc = pd.Series(y_test).value_counts()
top_labels = list(vc.head(min(12, len(vc))).index)
cm = confusion_matrix(y_test, pred_ens, labels=top_labels)
cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-8)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[classes[i][:12] for i in top_labels],
            yticklabels=[classes[i][:12] for i in top_labels])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Proposed Voting Ensemble', fontweight='bold')
plt.xticks(rotation=45, ha='right')
save_fig(f"{OUT_DIR}/confusion_matrix.png")
plt.show()

# %% [markdown]
# ##  Save Models

# %%
teacher.save(f"{OUT_DIR}/teacher.keras")
student.save(f"{OUT_DIR}/student.keras")
joblib.dump(ensemble, f"{OUT_DIR}/ensemble.joblib")

print(f"\n✅ All saved to: {OUT_DIR}")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"   - {f}")

# %% [markdown]

print("\n🎉 BENCHMARK COMPLETE!")

✅ XGBoost
✅ LightGBM
✅ CatBoost
✅ GPU available

✅ Setup complete!
📂 Mounting Google Drive...
Mounted at /content/drive
✅ Found 2 files already in local storage!
   - part-00028-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv (2.8 MB)
   - part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv (70.3 MB)
📋 Config: {'n_files_to_copy': 5, 'target_rows': 100000, 'min_class_count': 200, 'test_size': 0.2, 'batch_size': 256, 'epochs_dl': 25, 'epochs_kd': 15, 'patience': 5, 'kd_temp': 4.0, 'kd_alpha': 0.3, 'optuna_trials': 10}
📥 Need to copy files (have 2, need 5)
   Attempting to copy 3 files...
   [1/3] part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv...
       ✅ Done (64.5 MB)
   [2/3] part-00057-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv...
       ✅ Done (126.6 MB)
   [3/3] part-00112-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv...
       ✅ Done (74.7 MB)

✅ Successfully copied 3 new files

✅ Ready with 5 files

📂 Loading from 5 local files...
   Cap per class: 3225
   ✅ part-0

  0%|          | 0/10 [00:00<?, ?it/s]

   Best F1: 0.9372
   Acc: 0.9494 | F1: 0.9365 | MCC: 0.9472

🔄 Proposed Voting Ensemble
   Acc: 0.9528 | F1: 0.9441 | MCC: 0.9508

🔄 Proposed KD Student
Epoch 1/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: 2.9938
Epoch 2/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.7967
Epoch 3/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6333
Epoch 4/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5568
Epoch 5/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5090
Epoch 6/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4751
Epoch 7/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4496
Epoch 8/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4303
Epoch 9/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4126
Epoch 10/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3988
Epoch 11/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3886
Epoch 12/15
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3758
Epoch 13/15
202/202 ━━━━━━━━━━━━━━━━━━